# Coffee Crop Health — Fusion Agent
Combines Soil Agent (XGBoost) + Leaf Agent (EfficientNet-B3 Best) into one unified prediction.

| Agent | Input | Model | Accuracy |
|-------|-------|-------|----------|
| Soil Agent | 12 soil chemical readings | XGBoost + sample_weight | ~90% |
| Leaf Agent | Leaf photo | EfficientNet-B3 Best | 84.70% val |
| Fusion Agent | Both above | Rule-based fusion layer | Combined output |


## Step 1 — Install & Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')


Mounted at /content/drive
PyTorch : 2.10.0+cpu
CUDA    : False


## Step 2 — Paths & Config

In [2]:
import os

SOIL_MODEL_PATH = '/content/drive/MyDrive/Models/SoilAgent/soil_agent_v2.pkl'
LEAF_MODEL_PATH = '/content/drive/MyDrive/Models/LeafAgent/leaf_agent_swin_t.pth'
LEAF_META_PATH  = '/content/drive/MyDrive/Models/LeafAgent/leaf_agent_metadata.pkl'
TEST_DIR = '/content/drive/MyDrive/Datasets/Leaf Images_Dataset/Combined_Leaf_Dataset/final/test'

print('Checking model files...')
all_found = True
for path, name in [
    (SOIL_MODEL_PATH, 'Soil Agent (XGBoost)'),
    (LEAF_MODEL_PATH, 'Leaf Agent (Swin-T — 97.58% val)'),
    (LEAF_META_PATH,  'Leaf Metadata'),
]:
    found = os.path.exists(path)
    print(f'  {"OK" if found else "MISSING"}  {name}')
    print(f'         {path}')
    if not found:
        all_found = False

print()
print('All files found. Proceed.' if all_found else 'Fix missing files before continuing.')


Checking model files...
  OK  Soil Agent (XGBoost)
         /content/drive/MyDrive/Models/SoilAgent/soil_agent_v2.pkl
  OK  Leaf Agent (Swin-T — 97.58% val)
         /content/drive/MyDrive/Models/LeafAgent/leaf_agent_swin_t.pth
  OK  Leaf Metadata
         /content/drive/MyDrive/Models/LeafAgent/leaf_agent_metadata.pkl

All files found. Proceed.


## Step 3 — Load Soil Agent

In [3]:
import pickle
import numpy as np

with open(SOIL_MODEL_PATH, 'rb') as f:
    soil_bundle = pickle.load(f)

soil_model  = soil_bundle['model']
soil_scaler = soil_bundle['scaler']

SOIL_FEATURES = ['N', 'P', 'K', 'pH', 'EC', 'OC', 'S', 'Zn', 'Fe', 'Cu', 'Mn', 'B']
SOIL_CLASSES  = {0: 'Low Fertility', 1: 'Medium Fertility', 2: 'High Fertility'}

print('Soil Agent loaded')
print(f'  Model type : {type(soil_model).__name__}')
print(f'  Features   : {SOIL_FEATURES}')
print(f'  Classes    : {SOIL_CLASSES}')


Soil Agent loaded
  Model type : CalibratedClassifierCV
  Features   : ['N', 'P', 'K', 'pH', 'EC', 'OC', 'S', 'Zn', 'Fe', 'Cu', 'Mn', 'B']
  Classes    : {0: 'Low Fertility', 1: 'Medium Fertility', 2: 'High Fertility'}


## Step 4 — Load Leaf Agent (EfficientNet-B3 Best)

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image

with open(LEAF_META_PATH, 'rb') as f:
    leaf_meta = pickle.load(f)

CLASS_NAMES = leaf_meta['class_names']
MEAN        = leaf_meta.get('mean', [0.485, 0.456, 0.406])
STD         = leaf_meta.get('std',  [0.229, 0.224, 0.225])

IMG_SIZE    = 224
NUM_CLASSES = 9
DROPOUT     = 0.3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Rebuild Swin-T — must match Leaf_CNN_SwinT.ipynb exactly
leaf_model = models.swin_t(weights=None)
in_features = leaf_model.head.in_features   # 768

leaf_model.head = nn.Sequential(
    nn.Dropout(p=DROPOUT),
    nn.Linear(in_features, 512),
    nn.GELU(),
    nn.Dropout(p=0.15),
    nn.Linear(512, NUM_CLASSES)
)

state_dict = torch.load(LEAF_MODEL_PATH, map_location=device)
leaf_model.load_state_dict(state_dict)
leaf_model = leaf_model.to(device)
leaf_model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print('Leaf Agent loaded (Swin-T)')
print(f'  Device      : {device}')
print(f'  In features : {in_features}')
print(f'  Val accuracy: 97.58%')
print(f'  Classes     : {CLASS_NAMES}')


Leaf Agent loaded (Swin-T)
  Device      : cpu
  In features : 768
  Val accuracy: 97.58%
  Classes     : ['boron-B', 'calcium-Ca', 'healthy', 'iron-Fe', 'magnesium-Mg', 'manganese-Mn', 'nitrogen-N', 'phosphorus-P', 'potassium-K']


## Step 5 — Predict Functions

In [5]:
# Soil Agent predict
def predict_soil(soil_readings: dict) -> dict:
    """
    Input  : dict with 12 keys — N, P, K, pH, EC, OC, S, Zn, Fe, Cu, Mn, B
    Output : dict with fertility class, confidence, and status
    """
    values = np.array([[soil_readings[f] for f in SOIL_FEATURES]])
    scaled = soil_scaler.transform(values)
    probs  = soil_model.predict_proba(scaled)[0]
    pred   = int(np.argmax(probs))
    conf   = float(probs[pred])

    return {
        'predicted_class' : pred,
        'fertility_label' : SOIL_CLASSES[pred],
        'confidence_%'    : round(conf * 100, 1),
        'all_probs'       : {SOIL_CLASSES[i]: round(float(p)*100, 1)
                             for i, p in enumerate(probs)},
        'status'          : 'High confidence' if conf >= 0.60
                            else 'Low confidence — verify reading'
    }


# Leaf Agent predict
CONF_THRESHOLD = 0.60

def predict_leaf(image_path: str) -> dict:
    """
    Input  : path to a leaf image file
    Output : dict with deficiency class, confidence, top3, and status
    """
    img = Image.open(image_path).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = leaf_model(inp)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]

    top3_idx  = probs.argsort()[::-1][:3]
    pred_idx  = top3_idx[0]
    pred_name = CLASS_NAMES[pred_idx]
    conf      = float(probs[pred_idx])

    return {
        'predicted_class' : pred_name,
        'confidence_%'    : round(conf * 100, 1),
        'uncertainty_%'   : round((1 - conf) * 100, 1),
        'top_3'           : [{'class': CLASS_NAMES[i],
                               'prob_%': round(float(probs[i])*100, 1)}
                              for i in top3_idx],
        'status'          : 'High confidence' if conf >= CONF_THRESHOLD
                            else 'Low confidence — refer to agronomist'
    }

print('Predict functions ready')


Predict functions ready


## Step 6 — Fusion Logic

In [6]:
# Health status matrix
# Maps (soil_fertility_class, leaf_health_type) -> overall crop health status

HEALTH_MATRIX = {
    (2, 'healthy')     : 'Excellent',           # high fertility + healthy leaf
    (1, 'healthy')     : 'Good',                # medium fertility + healthy leaf
    (0, 'healthy')     : 'Monitor Soil',        # low fertility but leaf looks ok
    (2, 'deficient')   : 'Nutrient Imbalance',  # high fertility but leaf shows deficiency
    (1, 'deficient')   : 'Nutrient Deficiency', # medium fertility + leaf deficiency
    (0, 'deficient')   : 'Critical',            # both soil and leaf problematic
}

def fuse(soil_result: dict, leaf_result: dict) -> dict:
    """
    Combines soil and leaf predictions into one unified crop health report.
    Uses geometric mean of both confidences as overall confidence.
    Flags for expert review if either agent is below 60% confidence.
    """
    soil_cls  = soil_result['predicted_class']
    leaf_cls  = leaf_result['predicted_class']
    soil_conf = soil_result['confidence_%'] / 100
    leaf_conf = leaf_result['confidence_%'] / 100

    # Map leaf prediction to healthy / deficient
    leaf_type = 'healthy' if leaf_cls == 'healthy' else 'deficient'

    # Look up overall health
    overall = HEALTH_MATRIX.get((soil_cls, leaf_type), 'Unknown')

    # Overall confidence — geometric mean of both agents
    overall_conf = round((soil_conf * leaf_conf) ** 0.5 * 100, 1)

    # Flag low confidence agents
    low_conf = []
    if soil_conf < 0.60: low_conf.append('Soil Agent')
    if leaf_conf < 0.60: low_conf.append('Leaf Agent')

    return {
        'overall_health'      : overall,
        'needs_expert_review' : len(low_conf) > 0,
        'low_confidence_in'   : low_conf if low_conf else None,
        'overall_confidence_%': overall_conf,
        'soil' : {
            'fertility'    : soil_result['fertility_label'],
            'confidence_%' : soil_result['confidence_%'],
            'status'       : soil_result['status'],
            'all_probs'    : soil_result['all_probs'],
        },
        'leaf' : {
            'deficiency'   : leaf_result['predicted_class'],
            'confidence_%' : leaf_result['confidence_%'],
            'uncertainty_%': leaf_result['uncertainty_%'],
            'status'       : leaf_result['status'],
            'top_3'        : leaf_result['top_3'],
        }
    }

print('Fusion logic ready')
print()
print('Health matrix:')
for (s, l), status in HEALTH_MATRIX.items():
    print(f'  Soil={["Low","Medium","High"][s]:<8}  Leaf={l:<10}  ->  {status}')


Fusion logic ready

Health matrix:
  Soil=High      Leaf=healthy     ->  Excellent
  Soil=Medium    Leaf=healthy     ->  Good
  Soil=Low       Leaf=healthy     ->  Monitor Soil
  Soil=High      Leaf=deficient   ->  Nutrient Imbalance
  Soil=Medium    Leaf=deficient   ->  Nutrient Deficiency
  Soil=Low       Leaf=deficient   ->  Critical


## Step 7 — Single Sample Test

In [7]:
# Replace these with real values when testing on actual farm data

# Sample soil readings
sample_soil = {
    'N': 120, 'P': 45,  'K': 200,
    'pH': 6.2, 'EC': 0.8, 'OC': 1.2,
    'S': 15,  'Zn': 1.5, 'Fe': 8.0,
    'Cu': 0.5, 'Mn': 3.0, 'B': 0.4
}

# Pick one image from test set (replace with real leaf photo path)
import os
test_classes = sorted(os.listdir(TEST_DIR))
sample_cls   = test_classes[0]
sample_imgs  = os.listdir(os.path.join(TEST_DIR, sample_cls))
sample_leaf_path = os.path.join(TEST_DIR, sample_cls, sample_imgs[0])

print(f'Test class : {sample_cls}')
print(f'Image path : {sample_leaf_path}')
print()

# Run both agents
soil_result   = predict_soil(sample_soil)
leaf_result   = predict_leaf(sample_leaf_path)
fusion_result = fuse(soil_result, leaf_result)

# Print unified report
print('=' * 55)
print('     COFFEE CROP HEALTH ASSESSMENT REPORT')
print('=' * 55)
print(f"  Overall Health      : {fusion_result['overall_health']}")
print(f"  Overall Confidence  : {fusion_result['overall_confidence_%']}%")
if fusion_result['needs_expert_review']:
    print(f"  Expert Review       : YES — low confidence in {', '.join(fusion_result['low_confidence_in'])}")
else:
    print(f"  Expert Review       : Not required")
print()
print(f"  Soil Fertility      : {fusion_result['soil']['fertility']}")
print(f"  Soil Confidence     : {fusion_result['soil']['confidence_%']}%")
print(f"  Soil Probabilities  : {fusion_result['soil']['all_probs']}")
print()
print(f"  Leaf Deficiency     : {fusion_result['leaf']['deficiency']}")
print(f"  Leaf Confidence     : {fusion_result['leaf']['confidence_%']}%")
print(f"  Leaf Uncertainty    : {fusion_result['leaf']['uncertainty_%']}%")
print(f"  Leaf Top 3          : {fusion_result['leaf']['top_3']}")
print('=' * 55)


Test class : boron-B
Image path : /content/drive/MyDrive/Datasets/Leaf Images_Dataset/Combined_Leaf_Dataset/final/test/boron-B/B (54).jpg



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


     COFFEE CROP HEALTH ASSESSMENT REPORT
  Overall Health      : Critical
  Overall Confidence  : 87.6%
  Expert Review       : Not required

  Soil Fertility      : Low Fertility
  Soil Confidence     : 84.5%
  Soil Probabilities  : {'Low Fertility': 84.5, 'Medium Fertility': 12.8, 'High Fertility': 2.7}

  Leaf Deficiency     : boron-B
  Leaf Confidence     : 90.8%
  Leaf Uncertainty    : 9.2%
  Leaf Top 3          : [{'class': 'boron-B', 'prob_%': 90.8}, {'class': 'potassium-K', 'prob_%': 1.6}, {'class': 'iron-Fe', 'prob_%': 1.4}]


## Step 8 — Batch Test Across All 9 Classes

In [8]:
import random

print(f'{"True Class":<15} {"Leaf Pred":<15} {"Leaf Conf%":<12} {"Soil":<18} {"Overall Health":<22} {"Match"}')
print('-' * 90)

correct = 0
total   = 0

for cls_name in sorted(os.listdir(TEST_DIR)):
    cls_dir = os.path.join(TEST_DIR, cls_name)
    if not os.path.isdir(cls_dir):
        continue
    images   = os.listdir(cls_dir)
    img_path = os.path.join(cls_dir, random.choice(images))

    leaf_res   = predict_leaf(img_path)
    soil_res   = predict_soil(sample_soil)
    fusion_res = fuse(soil_res, leaf_res)

    match = 'YES' if cls_name == leaf_res['predicted_class'] else 'NO'
    if match == 'YES': correct += 1
    total += 1

    print(f"{cls_name:<15} {leaf_res['predicted_class']:<15} {leaf_res['confidence_%']:<12} {fusion_res['soil']['fertility']:<18} {fusion_res['overall_health']:<22} {match}")

print()
print(f'Batch accuracy (1 sample per class): {correct}/{total}')
print()
print('Note: batch uses 1 random image per class — run multiple times to see variation')


True Class      Leaf Pred       Leaf Conf%   Soil               Overall Health         Match
------------------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


boron-B         boron-B         89.2         Low Fertility      Critical               YES


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


calcium-Ca      calcium-Ca      74.3         Low Fertility      Critical               YES


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


nitrogen-N      nitrogen-N      92.4         Low Fertility      Critical               YES


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


phosphorus-P    phosphorus-P    92.3         Low Fertility      Critical               YES
potassium-K     potassium-K     88.8         Low Fertility      Critical               YES

Batch accuracy (1 sample per class): 8/9

Note: batch uses 1 random image per class — run multiple times to see variation


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## Step 9 — Save Fusion Bundle

In [9]:
import pickle, os

FUSION_SAVE_PATH = '/content/drive/MyDrive/Models/FusionAgent/fusion_agent.pkl'
os.makedirs(os.path.dirname(FUSION_SAVE_PATH), exist_ok=True)

fusion_bundle = {
    'soil_model'      : soil_model,
    'soil_scaler'     : soil_scaler,
    'soil_features'   : SOIL_FEATURES,
    'soil_classes'    : SOIL_CLASSES,
    'leaf_class_names': CLASS_NAMES,
    'leaf_img_size'   : IMG_SIZE,
    'leaf_mean'       : MEAN,
    'leaf_std'        : STD,
    'health_matrix'   : HEALTH_MATRIX,
    'conf_threshold'  : CONF_THRESHOLD,
    'leaf_model_path' : LEAF_MODEL_PATH,  # weights kept separate (too large for pkl)
    'version'         : 'fusion_v1_swint',
}

with open(FUSION_SAVE_PATH, 'wb') as f:
    pickle.dump(fusion_bundle, f)

print(f'Fusion bundle saved to {FUSION_SAVE_PATH}')
print()
print('Contents saved:')
for k in fusion_bundle:
    print(f'  {k}')
print()
print('Note: leaf model weights (.pth) are stored separately at LEAF_MODEL_PATH')
print('      Load them with torch.load() when deploying.')


Fusion bundle saved to /content/drive/MyDrive/Models/FusionAgent/fusion_agent.pkl

Contents saved:
  soil_model
  soil_scaler
  soil_features
  soil_classes
  leaf_class_names
  leaf_img_size
  leaf_mean
  leaf_std
  health_matrix
  conf_threshold
  leaf_model_path
  version

Note: leaf model weights (.pth) are stored separately at LEAF_MODEL_PATH
      Load them with torch.load() when deploying.
